### Setup

In [1]:
from openai import OpenAI
from PIL import Image
import base64

### Secure API Key Loading

In [2]:
with open("../api_key.txt", "r") as f:
    API_KEY = f.read().strip()

client = OpenAI(api_key=API_KEY)


### Create a Tiny Multimodal Evaluation Dataset

In [3]:
evaluation_dataset = [
    {
        "id": "vision_1",
        "type": "vision",
        "image_url": "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg",
        "question": "What animal is shown in the image?",
        "ground_truth": "cat"
    },
    {
        "id": "doc_1",
        "type": "document",
        "text": "Invoice Total: $250",
        "question": "What is the total amount?",
        "ground_truth": "$250"
    },
    {
        "id": "cross_modal_1",
        "type": "cross_modal",
        "image_url": "https://upload.wikimedia.org/wikipedia/commons/8/81/Stop_sign.png",
        "text": "This sign means vehicles must come to a complete halt.",
        "question": "What action should a driver take?",
        "ground_truth": "start"
    }
]

### Helper: Encode Image for Vision Input

In [4]:
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


### Run Multimodal Evaluation

In [5]:
results = []

for sample in evaluation_dataset:

    # Vision-only
    if sample["type"] == "vision":
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": sample["question"]},
                        {
                            "type": "image_url",
                            "image_url": {"url": sample["image_url"]}
                        }
                    ]
                }
            ]
        )
        answer = response.choices[0].message.content

    # Document-only
    elif sample["type"] == "document":
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": sample["text"] + "\n\n" + sample["question"]
                }
            ]
        )
        answer = response.choices[0].message.content

    # Cross-modal
    elif sample["type"] == "cross_modal":
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": sample["text"]},
                        {"type": "text", "text": sample["question"]},
                        {
                            "type": "image_url",
                            "image_url": {"url": sample["image_url"]}
                        }
                    ]
                }
            ]
        )
        answer = response.choices[0].message.content

    # Simple accuracy check
    correct = sample["ground_truth"].lower() in answer.lower()

    results.append({
        "id": sample["id"],
        "type": sample["type"],
        "correct": correct,
        "model_answer": answer
    })

### Compute Accuracy Metrics

In [6]:
total = len(results)
correct = sum(r["correct"] for r in results)

accuracy = correct / total

print("📊 Evaluation Results")
print("Total Samples:", total)
print("Correct:", correct)
print("Accuracy:", round(accuracy, 2))

📊 Evaluation Results
Total Samples: 3
Correct: 2
Accuracy: 0.67


### Summarize Results by Modality

In [7]:
from collections import defaultdict

by_type = defaultdict(list)

for r in results:
    by_type[r["type"]].append(r["correct"])

print("\n📈 Accuracy by Modality")
for modality, scores in by_type.items():
    print(modality, ":", round(sum(scores) / len(scores), 2))


📈 Accuracy by Modality
vision : 1.0
document : 1.0
cross_modal : 0.0
